# 【  維護操作手冊 — 銀行帳務查詢系統模組整合 】

---
程式模組分為三段落，分別為「網銀記錄自動化填寫至流水帳」、「憑證比對交易」、「查詢帳務助理」。預期可解決以下痛點：

> ***現階段痛點***

*   重複視窗操作導致審核時欄位誤判 — 作業易亂、眼疲勞高風險：將各銀行帳戶的對帳單/網銀逐筆資料，人工核對到每月份的各銀行帳戶流水帳工作表。
*   憑證與交易資料人工勾對 — 耗時、易錯、無法自動標註：逐筆點擊每月份各筆憑證，並人工比對上一步已填入交易資料的流水帳中，是否有和該憑證「日期、金額」相符的交易記錄，若有，需手動填寫憑證編號。
*   模糊交易查找流程破碎，視窗跳轉降低效率：當使用者不確定該筆交易得確切客戶/供應商，或是金額與日期等條件較為模糊時，若逐筆到流水帳查找，缺乏效率與不便性，且要開啟其他視窗才能找到，無法在同一介面操作。

---


> ***執行結果概述***

1.   解析玉山與兆豐銀行的 PDF 對帳單，擷取交易資料，並匯入至對應的 Excel 流水帳工作表。

  *  📦 程式架構拆解與執行流程
      -   正則式定義與預處理工具
          -   定義用於擷取金額、日期、時間的正則式：`MONEY_RE, DATE_RE, TIME_RE`
          -   `recover_money_tokens()` 提供備援方法：若 token 被錯誤拆分，可重新組合提取金額

      -   各銀行資料解析函式
          -   `parse_yushan(lines)`：針對玉山 187 對帳單 PDF 明細做欄位解析，擷取「帳務日期、摘要、提、存、餘額、備註、轉出入銀行代號/帳號」等欄位資料。
          -   `parse_mega(lines)`：針對兆豐 703 對帳單 PDF 明細做欄位解析，擷取「帳務日期、摘要、支出金額、存入金額、帳戶餘額、備註」等欄位資料。

      -   主函式 `匯入(目標月份=3)`
          - 上傳 Excel 初始模組工作表 與 PDF 對帳單/交易明細電子檔
          - 根據 PDF 檔名判斷銀行別（玉山 187 or 兆豐 703 or 其他帳戶）
          - 逐頁抽取 PDF 文字，傳給**對應解析函式產生 DataFrame**
          - 篩選對帳單/交易明細中的指定月份交易，並轉換日期型別，以及**將不同銀行提供的對帳單/交易明細欄位標題，標準化為與 Excel 工作表欄位標題一致**
          - 將原 Excel 中資料清空，再寫入更新後的新交易資料，欄位依序填入：帳務日期、客戶、摘要、支出、收入、餘額
          - 儲存並下載**已逐筆自動填寫交易記錄**的 Excel 檔案 `out.xlsx`
  *   ✅ 可達成的成果效益
      -   **自動解析 PDF 對帳單/交易明細**格式，大幅降低人工核對及輸入負擔
 	    -   自動**依月份篩選**資料，提高準確度與效率
	    -   直接匯入 Excel **相對應銀行帳戶代號**的工作表，並自動下載更新檔案，後續人工僅需 double check，且可執行後續 SOP
  *   🚧 可優化的建議方向
      -   PDF 抽取容錯設計
          -   **現狀問題**： 在匯入流程中使用 `pdfplumber.extract_text()` 解析 PDF 頁面內容。但如果某頁為空白、解析失敗或沒有可辨識文字，系統目前沒有記錄是哪一頁錯誤，僅略過。
          -   **風險**：使用者無法得知是否有某些頁面遺漏 → 容易錯過交易資料
          -   **改善建議**： 在每一頁解析時加上「頁面檢查與錯誤提示」，例如:

 	    -   Excel 欄位 mapping 由 config 控制
          -   **現狀問題**： 寫入 Excel 工作表時，程式直接指定欄位順序，若有帳戶格式不同（欄位順序或名稱不一致），就會：
          -   **風險**：需要手動調整程式碼、無法快速因應新銀行格式加入
          -   **改善建議**： 使用**「欄位對應字典」或「YAML 設定檔』**，動態決定每一工作表要填入哪些欄位、對應到哪一欄。例如：